## Stage 1 - Summary

# DINOv2: Learning Robust Visual Features without Supervision
**Authors:** Maxime Oquab, Timothée Darcet, Théo Moutakif, et al. (Meta AI, 2023)

## 1. Description
DINOv2 represents a milestone in the shift toward **Computer Vision Foundation Models**. DINOv2 proves that visual models can be trained entirely through self-supervision on augmented uncurated images , provided the data is intelligently filtered and the training architecture is stabilized for massive scale. 

The researchers developed an automated pipeline that starts with a small curated dataset and use it to trasnform a massive 1,2 Billion web scraped unique images into a high-quality dataset of 142 million images. By combining image-level and patch-level objectives, they produced a "universal" backbone. These features are so robust that the model can outperform supervised counterparts on tasks it was never explicitly trained for, such as monocular depth estimation and semantic segmentation, using only a simple linear layer on top of "frozen" features.

## 2. Main Ideas & Achievements
### 2.1 **Automatic Data Curation:** 
They have 2 sets of data: 

- Curated data (contains ImageNet-22k, the train split of ImageNet-1k, Google Landmarks and several fine-grained datasets)
- Uncurated web data (1,2 billion images) - scrape url links of images (from a publicly available repository of crawled web data) from `<img>` tags and filter for unsafe/restricted domains (dedup, nsfw, blurring etc.)

<img src="Images/data_pipeline_curation.png" width=800>

Goal is to expand the curated data with the uncurated data in a good/reliable way.

Their pipeline does the following:
- Embedding the images: 
    - Maps all raw images (both sets) into embeddings with a pre-trained model (self-supervised ViT-H/16 network pretrained on ImageNet-22k)
- Deduplication (Strict Threshold): 
    - Removes exact or near-exact mathematical matches (e.g., identical stock photos) between the uncurated web images. This reduces dataset size and prevents the model from wasting capacity memorizing exact copies.
- Matching/Retrieval (Broader Threshold): 
    - Matches the curated images to the uncurated images using the pre-trained model  based on semantic similarity, not visual identity. If the curated set has a dog, it pulls different photos of dogs (new poses, lighting, backgrounds) from the web data.
- Merge: 
    - By merging the original curated images with the matched web images, they create a vastly expanded dataset.

It takes less than two days to produce the LVD-142M dataset

### 2.2 **Discriminative Self-Supervision:** 
They combined the strengths of **DINO** (understanding the whole image) and **iBOT** (understanding local parts/patches).

They use a teacher student network method of training. Where the teacher is the Exponential Moving Average of the past students (So the latest model learns from averages of its past self):
- $ L_{DINO} = -\sum_{k=1}^{K} p_t^{(k)} \log p_s^{(k)} $ 
    - They use this loss function to make the model learn Object-level features. The student only sees the ear of the cat, while the teacher sees the whole cat. Then cross entropy of the 2 and we get a loss that forces the model to learn the relationship
- $ L_{iBOT} = -\sum_{i \in \text{masked}} \sum_{k=1}^{K} p_{t,i}^{(k)} \log p_{s,i}^{(k)} $ 
    - They use this loss function to make the model learn Patch-Level features. The student sees the whole image, but with a missing patch somewhere, while the teacher sees the whole image. Then cross entropy of the 2 and we get a loss that forces the model to learn the relationship
- $ L_{koleo} = -\frac{1}{n} \sum_{i=1}^{n} \log(d_{n,i}) \quad \text{where} \quad d_{n,i} = \min_{j \neq i} \|x_i - x_j\|_2 $
    - Where $d_{n,i}$ is the distance to the nearest neighbor.
        - $d_{n,i}$ is small: This means image $i$ is very close to another image $j$ in the batch.
        - The Logarithm: If $d_{n,i}$ is tiny, $\log(d_{n,i})$ is a large negative number.
        - The Negative Sign: The negative sign at the front flips that large negative number into a high loss.
    - The Result: To lower the loss, the student model is forced to push every vector as far away from its closest neighbor as possible.

Additional Key Techical components:
- **Resolution Finetuning**: Most training happens at a lower resolution to save time, but they add a short "burst" of training at 518x518 resolution at the end. This teaches the model to see the fine details necessary for tasks like pixel-perfect segmentation.
- **Multi-Objective Heads**: Unlike previous versions, they "untie" the weights for the global (DINO) and local (iBOT) tasks. At a 1-billion-parameter scale, they found the model performs better if it has separate "brains" for these two tasks.

### 2.3 **Efficient Implementation:**

To train a 1-billion-parameter model (**ViT-g**) on 142 million images, the authors implemented several hardware-level optimizations to overcome memory bottlenecks and speed up the training loop.

- **FlashAttention & xFormers:** They utilized memory-efficient attention kernels. This optimizes the self-attention mechanism by reducing memory complexity, allowing the model to handle larger batch sizes and higher resolutions without crashing the GPU VRAM.
- **Fully Sharded Data Parallel (FSDP):** To manage a 1B parameter model, they split the model weights, gradients, and optimizer states across all available GPUs. This "sharding" ensures no single GPU is overloaded, enabling the training of models that are physically larger than the memory of a single card.
- **Stochastic Depth:** Randomly "drops" or skips layers during the training pass. This acts as a powerful regularizer and reduces the computational cost per iteration.
- **Model distillation:**  For smaller models, they distill them from our largest model,the ViT-g, instead of training them from scratch

These improvements makes the approach around 2× faster and require 3× less memory than similar discriminative self-supervised methods, allowing them to leverage longer training with larger batch sizes.

That means:
- Faster Training per Epoch
- A larger batch provides a more accurate estimate of the "true" gradient for the entire dataset. This leads to a smoother, more deterministic convergence path with less "noise" compared to the erratic jumps seen with small batches.
- Fewer Weight Updates

### 2.4 **Ablation Studies:**
The authors performed a couple of abltion studies:
- **Incremental ablation:** (build-up) approach, starting from a basic iBOT baseline and adding features one by one to see how the performance climbed.
    - <img src="Images/ablation_table.png" width=400>
    - We can see how each technique impacts the final performance. They are additivie, everything above the line you are looking at is included so far
- **Curated vs. Uncurated:** They found that training on the **LVD-142M** (curated + retrieved web data) significantly outperformed training on just raw web data or just curated data. This proves that their "Retrieval" pipeline successfully kept the *diversity* of the web while maintaining the *quality* of curated sets.
- **Knowledge Distillation:** For small architectures, they distilled larger models instead of training them from scratch. Here is the comparisson
    - <img src="Images/knowledge_seg.png" width=600>
- **Impact of Resolution:** They tested training on smaller reslution first then on bigger in the end and gives almost the same as if only on the bigger
    - <img src="Images/res_ablation.png" width=600>
- **Loss Components:** Confirmed importance of KoLeo and iBOT losses
    - <img src="Images/ibot_ablation.png" width=600> 





### 3. **Results & Future Work**

#### **Notable Results: Closing the Gap**
DINOv2 is the first self-supervised learning (SSL) work to produce visual features that close the performance gap with (weakly) supervised models (like CLIP) across nearly all benchmarks without any task-specific finetuning. 

Key takeaways from the results:
* **"Frozen" Feature Power:** The information learned by the model is "readily available." Using a classifier as simple as a **linear layer**, the 1B parameter model achieves **86.5% on ImageNet-1k**, proving that the model understands object categories natively.
* **Emergent Properties:** Several high-level properties "emerge" from the scaling of data and parameters:
    * **Object Part Understanding:** The model can identify and segment specific parts of an object (e.g., the wing of a bird vs. the beak) without ever being told they are different parts.
    * **Geometry & Depth:** Despite only being shown 2D images, the model develops a deep understanding of scene geometry and relative distance (monocular depth estimation).


#### **Future Work: The "Visual Token" Vision**

This paper also demonstrateed that these visual features are compatible with classifiers as simple as linear layers - meaning the underlying information is readily available. 

In future work, they plan to leverage this ability to train a language-enabled AI system that can process visual features as if they were word tokens, and extract the required information to ground the system.